# Phase 6: Virtual 500-Panel Solar Farm

This notebook demonstrates the deterministic virtual topology, real-data-backed telemetry, controlled fault injection, and validation artifacts. It does not retrain Phases 3-5 models or add application infrastructure.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.simulation.farm_generator import generate_virtual_farm
from src.simulation.simulator_config import SimulationConfig

artifacts = generate_virtual_farm(SimulationConfig(), project_root)
panels = pd.read_csv(artifacts['panels'])
telemetry = pd.read_csv(artifacts['telemetry'])
events = pd.read_csv(artifacts['events'])
report = json.loads(artifacts['report'].read_text())
print(report)

{'panel_count': 500, 'rows': 20, 'columns': 25, 'panel_id_range': ['P0001', 'P0500'], 'string_count': 5, 'telemetry_record_count': 48000, 'time_range': ['2025-01-01T00:00:00+00:00', '2025-01-01T23:45:00+00:00'], 'fault_event_count': 4, 'fault_distribution': {'DEGRADATION': 48, 'NORMAL': 47860, 'OPEN_CIRCUIT': 33, 'SHADING': 27, 'SHORT_CIRCUIT': 32}, 'panel_efficiency_statistics': {'count': 500.0, 'mean': 0.999482, 'std': 0.034413, 'min': 0.940652, '25%': 0.968318, '50%': 1.000107, '75%': 1.029669, 'max': 1.059893}, 'dataset_source': 'clayton-h-costa/pv_fault_dataset via Phase 1 clean_pv_data.csv', 'random_seed': 42, 'simulation_configuration': {'seed': 42, 'rows': 20, 'columns': 25, 'telemetry_samples': 96, 'timestamp_start': '2025-01-01T00:00:00+00:00', 'sample_interval_minutes': 15, 'nominal_panel_power_w': 330.0, 'efficiency_min': 0.94, 'efficiency_max': 1.06, 'measurement_noise_std': 0.005, 'source_dataset': 'data/processed/clean_pv_data.csv', 'output_dir': 'data/processed', 'repor

## Topology and panel efficiency

The panel table contains 20 rows, 25 columns, five deterministic strings, and small efficiency variation around the configured nominal value.

In [2]:
display(panels.head())
display(panels.groupby('string_id').size())
display(panels['efficiency_factor'].describe())
display(events)
display(telemetry[telemetry['panel_id'] == 'P0103'][['timestamp', 'actual_power_w', 'fault_type', 'fault_severity']].head(20))

,panel_id,row_index,column_index,string_id,nominal_power_w,efficiency_factor,status
0,P0001,1,1,S01,330.0,1.032875,HEALTHY
1,P0002,1,2,S01,330.0,0.992665,HEALTHY
2,P0003,1,3,S01,330.0,1.043032,HEALTHY
3,P0004,1,4,S01,330.0,1.023684,HEALTHY
4,P0005,1,5,S01,330.0,0.951301,HEALTHY


string_id
S01    100
S02    100
S03    100
S04    100
S05    100
dtype: int64

count    500.000000
mean       0.999482
std        0.034413
min        0.940652
25%        0.968318
50%        1.000107
75%        1.029669
max        1.059893
Name: efficiency_factor, dtype: float64

,event_id,panel_id,start_time,end_time,fault_type,severity
0,E0001,P0103,2025-01-01 02:00:00+00:00,2025-01-01 08:45:00+00:00,SHADING,0.45
1,E0002,P0237,2025-01-01 05:30:00+00:00,2025-01-01 17:30:00+00:00,DEGRADATION,0.25
2,E0003,P0412,2025-01-01 11:15:00+00:00,2025-01-01 19:30:00+00:00,OPEN_CIRCUIT,0.90
3,E0004,P0058,2025-01-01 14:30:00+00:00,2025-01-01 22:30:00+00:00,SHORT_CIRCUIT,0.70


,timestamp,actual_power_w,fault_type,fault_severity
102,2025-01-01 00:00:00+00:00,0.002874,NORMAL,0.00
602,2025-01-01 00:15:00+00:00,0.001962,NORMAL,0.00
1102,2025-01-01 00:30:00+00:00,1.249277,NORMAL,0.00
1602,2025-01-01 00:45:00+00:00,257.297392,NORMAL,0.00
2102,2025-01-01 01:00:00+00:00,167.219255,NORMAL,0.00
2602,2025-01-01 01:15:00+00:00,0.004407,NORMAL,0.00
3102,2025-01-01 01:30:00+00:00,0.002055,NORMAL,0.00
3602,2025-01-01 01:45:00+00:00,0.001653,NORMAL,0.00
4102,2025-01-01 02:00:00+00:00,20.653035,SHADING,0.45
4602,2025-01-01 02:15:00+00:00,140.554244,SHADING,0.45


## Validation

Fault injection changes only the event windows and preserves non-negative electrical and power values. Topology identifiers remain simulation metadata and are not part of the Phase 3-5 ML feature schemas.

In [ ]:
numeric = telemetry.select_dtypes('number')
print('panels:', len(panels))
print('telemetry records:', len(telemetry))
print('fault distribution:', telemetry['fault_type'].value_counts().to_dict())
print('finite numeric telemetry:', numeric.notna().all().all())
print('non-negative power:', (telemetry['actual_power_w'] >= 0).all())

panels: 500
telemetry records: 48000
fault distribution: {'NORMAL': 47860, 'DEGRADATION': 48, 'OPEN_CIRCUIT': 33, 'SHORT_CIRCUIT': 32, 'SHADING': 27}
finite numeric telemetry: True
non-negative power: True
